# External enrichment with SCB (Statistics Sweden)

This notebook is intentionally **separate** from the core MYH pipeline (`data_preparation.ipynb`, `pipeline.ipynb`).
The main project continues to work even if the SCB API is unavailable, has changed its table structure, or this notebook has never been run.

## Purpose

The MYH application data tells us *how many* applications were submitted per municipality and how many were approved.
What it does not tell us is *context*: is Stockholm getting many approvals because it is a large city, or because it genuinely has a high approval rate relative to its population?

By joining each municipality to SCB demographic data we can:
- Calculate **applications per 100 000 residents** (normalise for city size)
- Surface **employment rate** and **unemployment rate** so the dashboard can correlate YH education demand with local labour market conditions

## Data sources

| Source | Table | Measures | Years available |
|--------|-------|----------|-----------------|
| SCB PxWeb | `BE/BE0101/BE0101A/BefolkningNy` | Total population per municipality | 1968 – 2024 |
| SCB PxWeb | `AM/AM0210/AM0210A/ArbStatusAr` | Employment rate, unemployment rate (ages 20–64) | 2020 – 2025 |

No API key is required. SCB's PxWeb API is open and free. The rate limit is 30 requests per 10 seconds, which this notebook stays well within.

## Important notes

- Population data goes up to **2024**. Applications from 2025 will receive 2024 population figures when used in the dashboard (the API always takes the latest available year for each municipality).
- Labour market data starts in **2020**. Municipalities in the 2018 and 2019 data will have `null` for the rate columns.
- This notebook queries only the **179 municipalities** that appear in the MYH dataset, not all 290 Swedish municipalities.


## Output

Running this notebook produces one file:

```
external_enrichment/data/scb_municipality_context.csv
```

The file has one row per **municipality × year** combination and these columns:

| Column | Type | Description |
|--------|------|-------------|
| `municipality` | text | Swedish municipality name (matches `kommun` in applications table) |
| `municipality_code` | text | Official SCB 4-digit code, e.g. `0180` for Stockholm |
| `year` | int | The reference year for the SCB figures |
| `population` | int | Total population at year-end |
| `employment_rate_percent` | float | Share of the 20–64 age group that is employed (%) |
| `unemployment_rate_percent` | float | Share of the workforce aged 20–64 that is unemployed (%) |

The FastAPI `/stats/municipality-enrichment` endpoint reads this file and joins it to the live application data.
If the file is missing or the notebook has not been run, the endpoint still works — it just returns application metrics without the SCB context columns.


In [1]:
from __future__ import annotations

import json
import unicodedata
from io import StringIO
from pathlib import Path
from typing import Any
from urllib import request
from urllib.error import HTTPError, URLError

import pandas as pd


def find_project_root() -> Path:
    """Find the project root from common notebook launch locations."""
    current_directory = Path.cwd().resolve()
    candidates = [current_directory, *current_directory.parents]

    for candidate in candidates:
        if (candidate / "data" / "curated_applications.csv").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find data/curated_applications.csv. "
        "Open the notebook from the project folder or check that the curated CSV exists."
    )


BASE_DIR = find_project_root()
APPLICATIONS_PATH = BASE_DIR / "data" / "curated_applications.csv"
OUTPUT_DIR = BASE_DIR / "external_enrichment" / "data"
OUTPUT_PATH = OUTPUT_DIR / "scb_municipality_context.csv"

POPULATION_API_URL = (
    "https://api.scb.se/OV0104/v1/doris/sv/ssd/BE/BE0101/BE0101A/BefolkningNy"
)
LABOUR_MARKET_API_URL = (
    "https://api.scb.se/OV0104/v1/doris/sv/ssd/AM/AM0210/AM0210A/ArbStatusAr"
)

YEARS = ["2018", "2019", "2020", "2021", "2022", "2023", "2024", "2025"]
LABOUR_MARKET_YEARS = ["2020", "2021", "2022", "2023", "2024", "2025"]
LABOUR_MARKET_AGE_GROUP = "20-64 år"
REQUEST_TIMEOUT_SECONDS = 60


ENRICHMENT_LOG: list[dict[str, str]] = []


def log_event(step: str, level: str, message: str, detail: str = "") -> None:
    """Append one event to the enrichment log.

    level: "ok" | "warning" | "error"
    """
    ENRICHMENT_LOG.append(
        {"Step": step, "Level": level, "Message": message, "Detail": detail}
    )


log_event("setup", "ok", "Project paths resolved", str(BASE_DIR))

## Setup and helper functions

### Swedish character normalisation

SCB API labels are in Swedish and contain `å`, `ä`, `ö`. The helper `normalize_label` decomposes these to their ASCII base letters using Unicode NFKD decomposition so that string comparisons work without hardcoding the Swedish characters:

```
'Folkmängd'          → 'folkmangd'
'arbetslöshet'       → 'arbetsloshet'
'sysselsättningsgrad' → 'sysselsattningsgrad'
```

This means we can search for `'folkmangd'` and still match the API label `'Folkmängd'`.

### PxWeb CSV format (wide, not long)

The SCB PxWeb v1 API returns data in **wide format**: when you request multiple years, each year becomes its own column rather than a row value. For example, a population query returns:

```
region            | ålder       | Folkmängd 2018 | Folkmängd 2019 | ...
0180 Stockholm    | totalt ålder| 961551         | 974073         | ...
```

The prepare step later melts this to long format. The encoding is **ISO-8859-1** (Latin-1), not UTF-8.

### Municipality codes

SCB uses 4-digit numeric municipality codes (e.g. `0180` for Stockholm). The API always returns them prefixed in the region label (`'0180 Stockholm'`). The `clean_municipality_name` helper strips the code prefix so municipality names match the MYH dataset.


In [2]:
def show_enrichment_log() -> pd.DataFrame:
    """Render the enrichment log as a dataframe."""
    return pd.DataFrame(ENRICHMENT_LOG)


def describe_http_error(error: HTTPError) -> str:
    """Return SCB's error text when the API responds with an HTTP error."""
    try:
        response_text = error.read().decode("utf-8", errors="replace")
    except Exception:
        response_text = ""
    return f"HTTP {error.code} {error.reason}. {response_text}".strip()


def describe_api_error(error: Exception) -> str:
    """Create a readable error message for network/API failures."""
    if isinstance(error, HTTPError):
        return describe_http_error(error)
    return str(error)


def fetch_metadata(api_url: str, step: str) -> dict[str, Any] | None:
    """Download table metadata from a SCB PxWeb v1 endpoint.

    Returns None and logs an error instead of raising a traceback. This keeps the
    notebook behaviour close to data_preparation.ipynb: every step reports its
    status and the final log explains what happened.
    """
    try:
        with request.urlopen(api_url, timeout=REQUEST_TIMEOUT_SECONDS) as response:
            metadata = json.loads(response.read().decode("utf-8"))
    except (HTTPError, URLError, TimeoutError, OSError) as error:
        log_event(
            step,
            "error",
            "SCB metadata request failed",
            f"{api_url} | {describe_api_error(error)}",
        )
        return None

    log_event(step, "ok", "SCB metadata loaded", api_url)
    return metadata


def normalize_label(value: object) -> str:
    """Normalize SCB labels so matching does not depend on Swedish letters.

    Swedish characters are decomposed (NFKD) so that å/ä/ö become their ASCII
    base letters after the combining diacritics are stripped.
    Examples: 'Folkmängd' → 'folkmangd',  'arbetslöshet' → 'arbetsloshet'
    """
    text = str(value).strip().lower()
    decomposed = unicodedata.normalize("NFKD", text)
    ascii_text = decomposed.encode("ascii", "ignore").decode("ascii")
    return " ".join(ascii_text.split())


def variable_by_code(metadata: dict[str, Any], code: str) -> dict[str, Any]:
    """Return one metadata variable by code."""
    for variable in metadata["variables"]:
        if variable["code"] == code:
            return variable
    raise KeyError(f"SCB variable not found: {code}")


def codes_matching_texts(
    metadata: dict[str, Any],
    variable_code: str,
    wanted_texts: set[str],
) -> list[str]:
    """Return SCB codes whose labels match the requested text values."""
    variable = variable_by_code(metadata, variable_code)
    wanted_normalized = {normalize_label(text) for text in wanted_texts}
    return [
        code
        for code, text in zip(variable["values"], variable["valueTexts"], strict=True)
        if normalize_label(text) in wanted_normalized
    ]


def code_matching_text(
    metadata: dict[str, Any],
    variable_code: str,
    wanted_text: str,
) -> str | None:
    """Return one SCB code by label after ASCII-safe normalization."""
    matches = codes_matching_texts(metadata, variable_code, {wanted_text})
    if not matches:
        log_event(
            "metadata",
            "error",
            "Required SCB value not found",
            f"{variable_code}={wanted_text}",
        )
        return None
    return matches[0]


def code_starting_with_text(
    metadata: dict[str, Any],
    variable_code: str,
    wanted_prefix: str,
) -> str | None:
    """Return one SCB code whose label starts with a normalized prefix."""
    variable = variable_by_code(metadata, variable_code)
    normalized_prefix = normalize_label(wanted_prefix)
    for code, text in zip(variable["values"], variable["valueTexts"], strict=True):
        if normalize_label(text).startswith(normalized_prefix):
            return code
    log_event(
        "metadata",
        "error",
        "Required SCB prefix not found",
        f"{variable_code} starts with {wanted_prefix}",
    )
    return None


def column_matching_text(dataframe: pd.DataFrame, wanted_text: str) -> str | None:
    """Find a dataframe column by normalized label."""
    wanted_normalized = normalize_label(wanted_text)
    for column in dataframe.columns:
        if normalize_label(column) == wanted_normalized:
            return column
    log_event("prepare", "error", "Required column not found", wanted_text)
    return None


def municipality_code_map(metadata: dict[str, Any]) -> dict[str, str]:
    """Map Swedish municipality names to SCB municipality codes."""
    region = variable_by_code(metadata, "Region")
    mapping = {}
    for code, name in zip(region["values"], region["valueTexts"], strict=True):
        if len(code) == 4:
            mapping[name.strip()] = code
    return mapping


def post_scb_csv(api_url: str, query: list[dict[str, Any]], step: str) -> pd.DataFrame:
    """Run a SCB query and parse the CSV response into a dataframe.

    SCB PxWeb returns CSV data in ISO-8859-1 (Latin-1) encoding regardless of
    the request encoding. The response is wide-format: when multiple years are
    requested, each year becomes a separate column rather than a row.
    """
    payload = json.dumps(
        {"query": query, "response": {"format": "CSV"}},
        ensure_ascii=False,
    ).encode("utf-8")
    api_request = request.Request(
        api_url,
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST",
    )

    try:
        with request.urlopen(api_request, timeout=REQUEST_TIMEOUT_SECONDS) as response:
            # SCB PxWeb sends CSV in ISO-8859-1 (Latin-1), not UTF-8
            csv_text = response.read().decode("iso-8859-1")
    except (HTTPError, URLError, TimeoutError, OSError) as error:
        log_event(
            step,
            "error",
            "SCB CSV request failed",
            f"{api_url} | {describe_api_error(error)}",
        )
        return pd.DataFrame()

    result = pd.read_csv(StringIO(csv_text), sep=None, engine="python")
    log_event(step, "ok", f"SCB CSV downloaded ({len(result):,} rows)", api_url)
    return result


def clean_municipality_name(value: str) -> str:
    """Remove SCB numeric region codes from labels like '0180 Stockholm'."""
    text = str(value).strip()
    parts = text.split(" ", 1)
    if len(parts) == 2 and parts[0].isdigit():
        return parts[1]
    return text


def normalize_number(value: object) -> float | None:
    """Convert SCB number strings to float values."""
    if pd.isna(value):
        return None
    text = str(value).replace(" ", "").replace(",", ".")
    if text in {"", "..", "."}:
        return None
    return float(text)

## Step 1 — Load the MYH municipalities

We only request SCB data for municipalities that actually appear in the MYH application data.
This keeps the API queries small and avoids wasting the rate limit on municipalities that are irrelevant to the project.

Two values are excluded:
- `NaN` / empty strings — rows where no municipality was recorded in the source Excel
- `"Flera kommuner"` — a special MYH marker meaning the application spans multiple municipalities; it is not a real municipality name

The result is a Python set of ~179 Swedish municipality names used as the query scope for all SCB requests.


In [3]:
applications = pd.read_csv(APPLICATIONS_PATH, encoding="utf-8-sig")

myh_municipalities = (
    applications["kommun"]
    .dropna()
    .astype(str)
    .str.strip()
    .loc[lambda series: series.ne("") & series.ne("Flera kommuner")]
    .unique()
)

myh_municipality_names = set(myh_municipalities)
len(myh_municipality_names)

log_event("source", "ok", f"Loaded {len(myh_municipality_names)} MYH municipalities")

## Step 2 — Fetch population from SCB (BefolkningNy)

**Table:** `BE/BE0101/BE0101A/BefolkningNy` — Population by region, age, sex, and marital status.

We query with `Alder = "tot"` (totalt ålder — all ages summed) and leave sex and marital status unspecified. When those dimensions are omitted from the PxWeb query, SCB automatically returns the aggregate total, so we get one row per municipality with the full population count.

**Why population?**
Raw application counts favour large cities. Stockholm has ~1 million residents and will naturally have more YH applications than Kiruna with ~25 000 residents. Expressing applications per 100 000 residents gives a fair comparison of how actively each municipality participates in YH education, regardless of city size.

**Note on years:** BefolkningNy has data from 1968 through 2024. The year 2025 is excluded from the query because that data has not yet been published by SCB. Applications from 2025 will use 2024 population figures in the dashboard (the API picks the most recent available year for each municipality).


In [4]:
population_metadata = fetch_metadata(POPULATION_API_URL, "population metadata")
population_raw = pd.DataFrame()
population_municipality_codes: dict[str, str] = {}

if population_metadata is not None:
    try:
        population_municipality_codes = municipality_code_map(population_metadata)
        selected_population_codes = [
            population_municipality_codes[name]
            for name in sorted(myh_municipality_names)
            if name in population_municipality_codes
        ]

        population_content_code = code_matching_text(
            population_metadata,
            "ContentsCode",
            "Folkmangd",
        )
        population_years = [year for year in YEARS if year != "2025"]

        if not selected_population_codes:
            log_event(
                "population",
                "error",
                "No MYH municipalities matched SCB municipality codes",
            )
        elif population_content_code is None:
            log_event(
                "population", "error", "Population content code could not be resolved"
            )
        else:
            population_raw = post_scb_csv(
                POPULATION_API_URL,
                [
                    {
                        "code": "Region",
                        "selection": {
                            "filter": "item",
                            "values": selected_population_codes,
                        },
                    },
                    {
                        "code": "Alder",
                        "selection": {"filter": "item", "values": ["tot"]},
                    },
                    {
                        "code": "ContentsCode",
                        "selection": {
                            "filter": "item",
                            "values": [population_content_code],
                        },
                    },
                    {
                        "code": "Tid",
                        "selection": {"filter": "item", "values": population_years},
                    },
                ],
                "population",
            )
    except Exception as error:
        log_event("population", "error", "Population preparation failed", str(error))

population_raw.head()

,region,ålder,Folkmängd 2018,Folkmängd 2019,Folkmängd 2020,Folkmängd 2021,Folkmängd 2022,Folkmängd 2023,Folkmängd 2024
0,0114 Upplands Väsby,totalt ålder,45543,46786,47184,47820,49262,50110,50323
1,0120 Värmdö,totalt ålder,44397,45000,45566,46232,46457,46637,46635
2,0123 Järfälla,totalt ålder,78480,79990,81274,83170,85460,86330,88950
3,0125 Ekerö,totalt ålder,28308,28690,28879,29096,29123,28808,28910
4,0126 Huddinge,totalt ålder,111722,112848,113234,113951,114504,113920,114304


## Step 3 — Fetch labour market data from SCB (ArbStatusAr)

**Table:** `AM/AM0210/AM0210A/ArbStatusAr` — Labour market status by municipality, sex, age group, and birth region.

We request two measures in one query:
- `arbetslöshet` — unemployment rate as a percentage of the workforce
- `sysselsättningsgrad` — employment rate as a percentage of the working-age population

Filters applied:
- **Kön (sex):** `totalt` (code `1+2`) — both men and women combined
- **Ålder (age):** `20-64 år` — standard working-age definition used by SCB/Eurostat
- **Födelseregion (birth region):** `totalt` — all residents regardless of where they were born

**Why labour market context?**
A municipality with high unemployment might have greater demand for retraining programmes (YH education is explicitly designed to address labour market needs). A municipality with a strong job market might have more employer-driven applications. These correlations can surface interesting patterns in the dashboard.

**Note on years:** ArbStatusAr starts in **2020**. Municipalities in the 2018 and 2019 data will have `null` in the `employment_rate_percent` and `unemployment_rate_percent` columns. This is expected and documented in the output.


In [5]:
labour_metadata = fetch_metadata(LABOUR_MARKET_API_URL, "labour metadata")
labour_raw = pd.DataFrame()

if labour_metadata is not None:
    try:
        labour_municipality_codes = municipality_code_map(labour_metadata)
        selected_labour_codes = [
            labour_municipality_codes[name]
            for name in sorted(myh_municipality_names)
            if name in labour_municipality_codes
        ]

        unemployment_code = code_matching_text(
            labour_metadata, "ContentsCode", "arbetsloshet"
        )
        employment_rate_code = code_matching_text(
            labour_metadata, "ContentsCode", "sysselsattningsgrad"
        )
        total_sex_code = code_matching_text(labour_metadata, "Kon", "totalt")
        total_birth_region_code = code_matching_text(
            labour_metadata, "Fodelseregion", "totalt"
        )
        # Use LABOUR_MARKET_AGE_GROUP ("20-64 år") to find the matching age-range code
        age_group_code = code_starting_with_text(
            labour_metadata, "Alder", LABOUR_MARKET_AGE_GROUP
        )

        required_codes = [
            unemployment_code,
            employment_rate_code,
            total_sex_code,
            total_birth_region_code,
            age_group_code,
        ]

        if not selected_labour_codes:
            log_event(
                "labour",
                "error",
                "No MYH municipalities matched SCB municipality codes",
            )
        elif any(code is None for code in required_codes):
            log_event(
                "labour",
                "error",
                "One or more required BAS codes could not be resolved",
            )
        else:
            labour_raw = post_scb_csv(
                LABOUR_MARKET_API_URL,
                [
                    {
                        "code": "Region",
                        "selection": {
                            "filter": "item",
                            "values": selected_labour_codes,
                        },
                    },
                    {
                        "code": "Kon",
                        "selection": {"filter": "item", "values": [total_sex_code]},
                    },
                    {
                        "code": "Alder",
                        "selection": {"filter": "item", "values": [age_group_code]},
                    },
                    {
                        "code": "Fodelseregion",
                        "selection": {
                            "filter": "item",
                            "values": [total_birth_region_code],
                        },
                    },
                    {
                        "code": "ContentsCode",
                        "selection": {
                            "filter": "item",
                            "values": [unemployment_code, employment_rate_code],
                        },
                    },
                    {
                        "code": "Tid",
                        "selection": {"filter": "item", "values": LABOUR_MARKET_YEARS},
                    },
                ],
                "labour",
            )
    except Exception as error:
        log_event("labour", "error", "Labour-market preparation failed", str(error))

labour_raw.head()

,region,kön,ålder,födelseregion,arbetslöshet 2020,arbetslöshet 2021,arbetslöshet 2022,arbetslöshet 2023,arbetslöshet 2024,arbetslöshet 2025,sysselsättningsgrad 2020,sysselsättningsgrad 2021,sysselsättningsgrad 2022,sysselsättningsgrad 2023,sysselsättningsgrad 2024,sysselsättningsgrad 2025
0,0114 Upplands Väsby,totalt,20-64 år,totalt,6.9,7.2,5.5,5.5,6.1,6.7,78.4,78.4,80.0,80.0,79.6,79.1
1,0120 Värmdö,totalt,20-64 år,totalt,3.8,3.6,2.8,2.7,3.2,3.2,82.9,83.3,84.3,84.5,84.0,83.7
2,0123 Järfälla,totalt,20-64 år,totalt,7.3,7.4,6.2,5.8,6.4,7.1,78.3,78.6,79.6,79.8,79.4,78.7
3,0125 Ekerö,totalt,20-64 år,totalt,3.6,3.6,2.6,2.8,2.9,3.3,82.6,82.2,83.0,83.0,83.0,82.7
4,0126 Huddinge,totalt,20-64 år,totalt,6.8,7.0,5.5,5.2,5.7,6.0,77.3,77.5,79.1,79.3,78.8,78.4


## Step 4 — Prepare the final enrichment table

Both SCB responses arrive in **wide format** — years are column name suffixes rather than row values:

```
Population response:
  region           | Folkmängd 2018 | Folkmängd 2019 | ... | Folkmängd 2024

Labour market response:
  region           | arbetslöshet 2020 | ... | sysselsättningsgrad 2025
```

The prepare step converts these to **long format** (one row per municipality × year) using `melt`, then joins the two sources.

### Transformation steps

1. **Population melt** — find all columns with a trailing 4-digit year, melt to `(municipality, year, population)`
2. **Labour melt** — same pattern, then strip the year suffix from the column name to get the metric name, then `pivot_table` to put `unemployment_rate_percent` and `employment_rate_percent` side by side
3. **Municipality code join** — attach the official SCB 4-digit code for each municipality (e.g. `0180` for Stockholm)
4. **Left merge** — population is the primary source; labour market data is joined with `how="left"` so municipalities with no labour market match still appear in the output (with `null` rate values)

### Final schema

```
municipality  municipality_code  year  population  employment_rate_percent  unemployment_rate_percent
Stockholm     0180               2022  984748      79.4                     4.8
Stockholm     0180               2023  988943      79.4                     4.7
...
```


In [6]:
import re as _re

_YEAR_SUFFIX = _re.compile(r"\s*(\d{4})$")

municipality_context_columns = [
    "municipality",
    "municipality_code",
    "year",
    "population",
    "employment_rate_percent",
    "unemployment_rate_percent",
]


def _year_value_columns(dataframe: pd.DataFrame, exclude_region_col: str) -> list[str]:
    """Return columns whose name ends with a 4-digit year (SCB wide format)."""
    return [
        col
        for col in dataframe.columns
        if col != exclude_region_col and _YEAR_SUFFIX.search(col)
    ]


def _extract_year(label: str) -> int | None:
    """Extract the trailing 4-digit year from a wide-format column name."""
    match = _YEAR_SUFFIX.search(label)
    return int(match.group(1)) if match else None


def _strip_year(label: str) -> str:
    """Remove the trailing year from a wide-format column name."""
    return _YEAR_SUFFIX.sub("", label).strip()


if population_raw.empty:
    log_event(
        "prepare",
        "error",
        "Population data is missing, so no SCB context file can be prepared",
    )
    municipality_context = pd.DataFrame(columns=municipality_context_columns)
else:
    try:
        # ── Population: SCB PxWeb returns wide format ──────────────────────────────
        # When multiple years are requested, each year becomes its own column:
        # 'region', 'ålder', 'Folkmängd 2018', 'Folkmängd 2019', ..., 'Folkmängd 2024'
        # We melt these to long format: one row per municipality × year.
        population_region_column = column_matching_text(population_raw, "region")
        if population_region_column is None:
            raise ValueError("Population dataframe missing 'region' column")

        population_value_columns = _year_value_columns(
            population_raw, population_region_column
        )
        if not population_value_columns:
            raise ValueError(
                "Population dataframe has no year-suffixed columns (expected 'Folkmängd YYYY')"
            )

        population = (
            population_raw[[population_region_column] + population_value_columns]
            .melt(
                id_vars=[population_region_column],
                var_name="year_label",
                value_name="population",
            )
            .rename(columns={population_region_column: "municipality"})
        )
        population["municipality"] = population["municipality"].map(
            clean_municipality_name
        )
        population["year"] = population["year_label"].map(_extract_year)
        population = population.dropna(subset=["year"])
        population["year"] = population["year"].astype(int)
        population["population"] = population["population"].map(normalize_number)
        population = population[["municipality", "year", "population"]].copy()
        population["population"] = population["population"].astype("Int64")
        log_event(
            "prepare",
            "ok",
            f"Population data shaped ({len(population):,} municipality-year rows)",
        )

        # ── Labour market: same wide format ────────────────────────────────────────
        # SCB returns one row per municipality with year-labelled metric columns:
        # 'region', 'kön', 'ålder', 'födelseregion',
        # 'arbetslöshet 2020', ..., 'sysselsättningsgrad 2025'
        # We melt, extract year + metric name, then pivot to get two rate columns.
        if labour_raw.empty:
            log_event(
                "prepare",
                "warning",
                "Labour-market data is missing; exporting population-only context",
            )
            labour_wide = pd.DataFrame(
                columns=[
                    "municipality",
                    "year",
                    "employment_rate_percent",
                    "unemployment_rate_percent",
                ]
            )
        else:
            labour_region_column = column_matching_text(labour_raw, "region")
            if labour_region_column is None:
                raise ValueError("Labour dataframe missing 'region' column")

            labour_value_columns = _year_value_columns(labour_raw, labour_region_column)
            if not labour_value_columns:
                raise ValueError("Labour dataframe has no year-suffixed columns")

            labour = (
                labour_raw[[labour_region_column] + labour_value_columns]
                .melt(
                    id_vars=[labour_region_column],
                    var_name="metric_year_label",
                    value_name="value",
                )
                .rename(columns={labour_region_column: "municipality"})
            )
            labour["municipality"] = labour["municipality"].map(clean_municipality_name)
            labour["year"] = labour["metric_year_label"].map(_extract_year)
            labour = labour.dropna(subset=["year"])
            labour["year"] = labour["year"].astype(int)
            labour["value"] = labour["value"].map(normalize_number)
            # 'arbetslöshet 2022' → strip year → normalize → 'arbetsloshet'
            labour["metric_normalized"] = (
                labour["metric_year_label"].map(_strip_year).map(normalize_label)
            )

            labour_wide = labour.pivot_table(
                index=["municipality", "year"],
                columns="metric_normalized",
                values="value",
                aggfunc="first",
            ).reset_index()
            labour_wide.columns.name = None
            labour_wide = labour_wide.rename(
                columns={
                    "arbetsloshet": "unemployment_rate_percent",
                    "sysselsattningsgrad": "employment_rate_percent",
                }
            )
            # Guard: ensure both rate columns exist even if the pivot produced neither
            for rate_col in ("unemployment_rate_percent", "employment_rate_percent"):
                if rate_col not in labour_wide.columns:
                    labour_wide[rate_col] = None
            log_event(
                "prepare",
                "ok",
                f"Labour market data shaped ({len(labour_wide):,} municipality-year rows)",
            )

        # ── Municipality code lookup ────────────────────────────────────────────────
        # Map each Swedish municipality name to its 4-digit SCB code so the
        # dashboard can reference official identifiers alongside our own naming.
        municipality_codes = pd.DataFrame(
            [
                {"municipality": name, "municipality_code": code}
                for name, code in population_municipality_codes.items()
                if name in myh_municipality_names
            ]
        )

        municipality_context = (
            population.merge(labour_wide, on=["municipality", "year"], how="left")
            .merge(municipality_codes, on="municipality", how="left")
            .sort_values(["municipality", "year"])
            .reset_index(drop=True)
        )
        log_event(
            "prepare",
            "ok",
            f"Final table: {len(municipality_context):,} rows, {municipality_context['municipality'].nunique()} municipalities",
        )

    except Exception as error:
        log_event("prepare", "error", "Failed to prepare SCB context table", str(error))
        municipality_context = pd.DataFrame(columns=municipality_context_columns)

municipality_context.head(10)

,municipality,year,population,unemployment_rate_percent,employment_rate_percent,municipality_code
0,Alingsås,2018,41070,NaN,NaN,1489
1,Alingsås,2019,41420,NaN,NaN,1489
2,Alingsås,2020,41602,4.6,83.0,1489
3,Alingsås,2021,41853,4.3,83.5,1489
4,Alingsås,2022,42199,3.4,84.7,1489
5,Alingsås,2023,42382,3.3,85.3,1489
6,Alingsås,2024,42722,3.7,84.6,1489
7,Arboga,2018,14138,NaN,NaN,1984
8,Arboga,2019,14087,NaN,NaN,1984
9,Arboga,2020,14039,8.0,77.7,1984


## Step 5 — Export

Writes `scb_municipality_context.csv` to the `external_enrichment/data/` folder.

The file is **not committed to git** (covered by `.gitignore`). To update the enrichment data, simply re-run this notebook. The API will automatically pick up the new file on the next request to `/stats/municipality-enrichment`.

If population or labour market data could not be fetched, no file is written and the enrichment log below explains why.


In [7]:
if municipality_context.empty:
    log_event("export", "error", "No SCB context rows to export")
    print("No SCB context file was written. See ENRICHMENT_LOG below.")
else:
    try:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        municipality_context.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
        log_event(
            "export",
            "ok",
            f"CSV written ({len(municipality_context):,} rows)",
            str(OUTPUT_PATH),
        )
        print(f"Wrote {len(municipality_context):,} rows to {OUTPUT_PATH}")
    except OSError as error:
        log_event("export", "error", "Failed to write SCB context CSV", str(error))
        print("No SCB context file was written. See ENRICHMENT_LOG below.")

show_enrichment_log()

Wrote 1,246 rows to myh-data-pipeline\external_enrichment\data\scb_municipality_context.csv


,Step,Level,Message,Detail
0,setup,ok,Project paths resolved,myh-data-pipeline
1,source,ok,Loaded 179 MYH municipalities,
2,population metadata,ok,SCB metadata loaded,https://api.scb.se/OV0104/v1/doris/sv/ssd/BE/B...
3,population,ok,SCB CSV downloaded (178 rows),https://api.scb.se/OV0104/v1/doris/sv/ssd/BE/B...
4,labour metadata,ok,SCB metadata loaded,https://api.scb.se/OV0104/v1/doris/sv/ssd/AM/A...
5,labour,ok,SCB CSV downloaded (178 rows),https://api.scb.se/OV0104/v1/doris/sv/ssd/AM/A...
6,prepare,ok,"Population data shaped (1,246 municipality-yea...",
7,prepare,ok,"Labour market data shaped (1,068 municipality-...",
8,prepare,ok,"Final table: 1,246 rows, 178 municipalities",
9,export,ok,"CSV written (1,246 rows)",myh-data-pipeline\...


## Enrichment log

Every step appends a record to `ENRICHMENT_LOG`. Statuses follow the same convention as `data_preparation.ipynb`:

| Status | Meaning |
|--------|---------|
| `ok` | Step completed successfully |
| `warning` | Step completed with a non-fatal issue (e.g. labour data missing, exporting population only) |
| `error` | Step failed; output file may be incomplete or absent |

A successful run should show `ok` for every row. If any row shows `error`, read the `Detail` column to understand why.


In [8]:
pd.DataFrame(ENRICHMENT_LOG)

,Step,Level,Message,Detail
0,setup,ok,Project paths resolved,myh-data-pipeline
1,source,ok,Loaded 179 MYH municipalities,
2,population metadata,ok,SCB metadata loaded,https://api.scb.se/OV0104/v1/doris/sv/ssd/BE/B...
3,population,ok,SCB CSV downloaded (178 rows),https://api.scb.se/OV0104/v1/doris/sv/ssd/BE/B...
4,labour metadata,ok,SCB metadata loaded,https://api.scb.se/OV0104/v1/doris/sv/ssd/AM/A...
5,labour,ok,SCB CSV downloaded (178 rows),https://api.scb.se/OV0104/v1/doris/sv/ssd/AM/A...
6,prepare,ok,"Population data shaped (1,246 municipality-yea...",
7,prepare,ok,"Labour market data shaped (1,068 municipality-...",
8,prepare,ok,"Final table: 1,246 rows, 178 municipalities",
9,export,ok,"CSV written (1,246 rows)",myh-data-pipeline\...
